# Act — Tool Calling
### Pune AI Builders Meetup — Notebook 03

---

> **Where we left off:** We have a complete candidate profile.
> Now the counsellor needs to *act* — search jobs, verify GitHub, make recommendations.
>
> **The problem:** *Can the LLM just search jobs on its own?*

Let's find out.

In [2]:
import os, json
import pandas as pd

# Complete profile after Notebooks 01 + 02
ARJUN = {
    "name"              : "Arjun Mehta",
    "current_role"      : "Senior SDE-II at Flipkart",
    "total_experience"  : 8,
    "top_skills"        : ["Java", "Go", "Kafka", "Redis", "distributed systems"],
    "career_goal"       : "Senior IC or Tech Lead in fintech/infra",
    "salary_expectation": "42–50 LPA",
    "notice_period"     : "60 days",
    "reason_for_change" : "Work became predictable. Wants harder problems.",
    "gap_explanation"   : "First gap personal. Second intentional break — built review-gpt.",
    "open_to_relocation": "Bangalore preferred, open to remote",
    "github"            : "mahadevTW",
}

# Job database loaded from CSV
JOB_DATABASE = pd.read_csv("../data/jobs.csv")

print(f"Setup done. Loaded {len(JOB_DATABASE)} jobs from CSV.")
print(JOB_DATABASE[["job_title", "company_name", "skills", "experience_required"]].to_string(index=False))

Setup done. Loaded 500 jobs from CSV.
                                                                                                             job_title  company_name  skills  experience_required
                                                                                         Manager, Software Development        Fiserv     NaN                  NaN
                                                                            B2B Product Manager/Senior Product Manager        Fiserv     NaN                  NaN
                                                                             Advisor, Software Development Engineering        Fiserv     NaN                  NaN
                                                                                              Principal Engineer AI/ML         Adobe     NaN                  NaN
                                                                                               Sr. M&A Finance Analyst         Adobe     NaN            

---
## Step 1 — Ask the LLM Without Any Tools

Arjun mentions his GitHub: `github.com/arjun91`

> *"Can you check Arjun's GitHub and verify his technical claims?"*

In [3]:
OPENAI_API_KEY="sk-proj-your-openai-api-key-here"  # ← replace with your OpenAI API key
print(f"OpenAI API key set (not printed for security)")

OpenAI API key set (not printed for security)


In [4]:
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY))

In [5]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system",  "content": "You are a career counsellor."},
        {"role": "user",    "content": f"Candidate profile:\n{json.dumps(ARJUN, indent=2)}\n\nCheck Arjun's GitHub (mahadevTW) and verify his technical claims."}
    ]
)

print(resp.choices[0].message.content)

I'm unable to access external websites, including GitHub, and verify information or check user profiles directly. However, I can guide you on how to evaluate Arjun's technical claims based on his GitHub profile.

Here’s how you can review Arjun's GitHub to validate his skills:

1. **Check Repositories**: Look for repositories that showcase his work with Java, Go, Kafka, and Redis. He should have projects that reflect his expertise in distributed systems.

2. **Code Quality**: Evaluate the quality of the code in his repositories. Look for clear documentation, appropriate use of design patterns, and proper code structure, which indicate a good understanding of software development practices.

3. **Activity Level**: Check how active he is on GitHub. Frequent commits, contributions to projects, and participation in open-source projects can reflect his passion and ongoing learning in technology.

4. **Projects Related to Fintech/Infra**: Since his career goal is to move into a Senior IC or 

**The LLM admits it can't.**

It has no internet. No file access. No database connection.
On its own, the LLM can only **think** — it can never **act**.

```
LLM alone:        Think  →  Answer        (limited to what it was trained on)
LLM + Tools:      Think  →  Call Tool  →  Get Result  →  Answer
```

**Tools are how we give the LLM hands.**

---
## Step 2 — Define the Tools

In [6]:
import requests

def read_github(username: str) -> dict:
    """Fetch real GitHub profile via public API. Gracefully handles rate-limits."""
    demo_username = "mahadevTW"
    try:
        user  = requests.get(f"https://api.github.com/users/{demo_username}", timeout=5).json()
        repos = requests.get(f"https://api.github.com/users/{demo_username}/repos?sort=stars&per_page=10", timeout=5).json()
    except Exception as e:
        return {"error": str(e)}

    if not isinstance(user, dict) or "login" not in user:
        return {"error": user.get("message", "GitHub API unavailable") if isinstance(user, dict) else "GitHub API unavailable"}
    if not isinstance(repos, list):
        return {"error": repos.get("message", "Could not fetch repos") if isinstance(repos, dict) else "Could not fetch repos"}

    lang_count = {}
    for r in repos:
        if r.get("language"):
            lang_count[r["language"]] = lang_count.get(r["language"], 0) + 1
    return {
        "username"     : demo_username,
        "name"         : user.get("name"),
        "bio"          : user.get("bio"),
        "company"      : user.get("company"),
        "public_repos" : user.get("public_repos"),
        "followers"    : user.get("followers"),
        "top_languages": sorted(lang_count, key=lang_count.get, reverse=True)[:3],
        "top_repos"    : [{"name": r["name"], "stars": r["stargazers_count"],
                           "lang": r["language"], "desc": r["description"],
                           "last_pushed": r["pushed_at"][:10]} for r in repos[:5]],
    }


In [39]:
response  = read_github("mahadevTW")
print(json.dumps(response, indent=2))

{
  "username": "mahadevTW",
  "name": "Mahadev Vyavahare",
  "bio": null,
  "company": null,
  "public_repos": 70,
  "followers": 27,
  "top_languages": [
    "HTML",
    "Shell",
    "JavaScript"
  ],
  "top_repos": [
    {
      "name": "build-your-first-ai-native-app-public",
      "stars": 1,
      "lang": "Shell",
      "desc": null,
      "last_pushed": "2026-06-26"
    },
    {
      "name": "classroom-code",
      "stars": 0,
      "lang": "JavaScript",
      "desc": null,
      "last_pushed": "2026-06-02"
    },
    {
      "name": "rebind-rise",
      "stars": 0,
      "lang": "HTML",
      "desc": null,
      "last_pushed": "2026-06-26"
    },
    {
      "name": "nyfinity-pharma-website",
      "stars": 0,
      "lang": "HTML",
      "desc": null,
      "last_pushed": "2025-12-24"
    },
    {
      "name": "ecobeej-website",
      "stars": 0,
      "lang": "HTML",
      "desc": null,
      "last_pushed": "2025-12-22"
    }
  ]
}


In [7]:
def search_jobs(skills: list, domain: str = None, min_years: int = 0) -> list:
    """Search job database by matching candidate skills against job descriptions."""
    results = []
    active_jobs = JOB_DATABASE[JOB_DATABASE["is_active"] == True]

    for _, row in active_jobs.iterrows():
        # Build searchable text from title + description (skills column is empty in real data)
        searchable = " ".join([
            str(row.get("job_title", "") or ""),
            str(row.get("job_description", "") or ""),
        ]).lower()

        # Count how many candidate skills appear in the text
        matched = [s for s in skills if s.strip().lower() in searchable]
        overlap  = len(matched) / len(skills) if skills else 0

        if overlap >= 0.4:
            results.append({
                "id"          : row["id"],
                "job_uid"     : row["job_uid"],
                "job_title"   : row["job_title"],
                "company"     : row["company_name"],
                "location"    : row["location"],
                "salary"      : row.get("salary") or "Not disclosed",
                "matched_skills": matched,
                "match_score" : round(overlap * 100),
                "apply_link"  : row["apply_link"],
                "description" : str(row.get("job_description", ""))[:300] + "...",
            })

    return sorted(results, key=lambda x: x["match_score"], reverse=True)[:3]


# Quick sanity check
test = search_jobs(["Java", "Go", "Kafka", "Redis", "distributed systems"], min_years=8)
print(f"Sanity check — found {len(test)} jobs")
for j in test:
    print(f"  [{j['match_score']}%] {j['job_title']} @ {j['company']}  |  matched: {j['matched_skills']}")


Sanity check — found 3 jobs
  [80%] Manager, Software Development @ Fiserv  |  matched: ['Java', 'Go', 'Kafka', 'distributed systems']
  [80%] Software Engineer @ Intel  |  matched: ['Java', 'Go', 'Kafka', 'distributed systems']
  [80%] Senior Software Engineer - Python @ PayPal  |  matched: ['Go', 'Kafka', 'Redis', 'distributed systems']


In [8]:
# ── Tool dispatcher — maps LLM's tool name → Python function ─────────────
def execute_tool(name: str, args: dict):
    if name == "read_github":  return read_github(**args)
    if name == "search_jobs":  return search_jobs(**args)
    raise ValueError(f"Unknown tool: {name}")
# ── Tool schema — what we tell the LLM these tools can do ─────────────────


In [9]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_github",
            "description": "Fetch and analyse a candidate's public GitHub profile — repos, languages, activity.",
            "parameters": {
                "type": "object",
                "properties": {
                    "username": {"type": "string", "description": "GitHub username"}
                },
                "required": ["username"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_jobs",
            "description": "Search open roles matching the candidate's skills and experience.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skills"   : {"type": "array",  "items": {"type": "string"}, "description": "Candidate's skills"},
                    "domain"   : {"type": "string",  "description": "Preferred domain (e.g. fintech, infra)"},
                    "min_years": {"type": "integer", "description": "Years of experience"}
                },
                "required": ["skills", "min_years"]
            }
        }
    },
]

print(f"Registered {len(TOOLS)} tools: {[t['function']['name'] for t in TOOLS]}")

# Quick sanity check — pull live data right now
print("\n── Live GitHub check ──")
profile = read_github("mahadevTW")
print(f"  Name         : {profile['name']}")
print(f"  Company      : {profile['company']}")
print(f"  Public repos : {profile['public_repos']}  |  Followers: {profile['followers']}")
print(f"  Top languages: {profile['top_languages']}")
print(f"  Top repos    :")
for r in profile["top_repos"]:
    print(f"    ★ {r['stars']:>3}  [{r['lang'] or '?':>10}]  {r['name']} — {r['desc']}")

Registered 2 tools: ['read_github', 'search_jobs']

── Live GitHub check ──
  Name         : Mahadev Vyavahare
  Company      : None
  Public repos : 70  |  Followers: 28
  Top languages: ['HTML', 'Jupyter Notebook', 'JavaScript']
  Top repos    :
    ★   1  [Jupyter Notebook]  build-your-first-ai-native-app-public — None
    ★   0  [JavaScript]  classroom-code — None
    ★   0  [      HTML]  rebind-rise — None
    ★   0  [      HTML]  nyfinity-pharma-website — None
    ★   0  [      HTML]  ecobeej-website — None


---
## Step 4 — The Tool Calling

The LLM requests a tool. We execute it and send the result back. It continues.
Repeat until there are no more tool calls.
very IMP : **LLM does not act, it instructs ecosystem to act with required details**
```
LLM response
  ├── finish_reason = "tool_calls"  →  execute → send result back → repeat
  └── finish_reason = "stop"        →  return final answer
```

In [14]:
# One call — no loop, no execution. Just observe what comes back.
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a career counsellor."},
        {"role": "user",   "content":
            f"Candidate profile:\n{json.dumps(ARJUN, indent=2)}\n\n"
            "Check Arjun's GitHub (mahadevTW) and verify his technical claims."},
    ],
    tools=TOOLS,
)

msg = resp.choices[0].message

print(f"finish_reason : {resp.choices[0].finish_reason}")   # → "tool_calls"
print(f"content       : {msg.content}")                      # → None (nothing written yet)
print(f"tool_calls    : {len(msg.tool_calls)} request(s)\n")

for tc in msg.tool_calls:
    args = json.loads(tc.function.arguments)
    print(f"  ► Tool      : {tc.function.name}")
    print(f"    Arguments : {args}")
    print()

print("──────────────────────────────────────────────────────")
print("The LLM stopped here. It handed control back to us.")
print("WE decide whether to execute the tool. The LLM waits.")

finish_reason : tool_calls
content       : None
tool_calls    : 1 request(s)

  ► Tool      : read_github
    Arguments : {'username': 'mahadevTW'}

──────────────────────────────────────────────────────
The LLM stopped here. It handed control back to us.
WE decide whether to execute the tool. The LLM waits.


Now that we've seen what the LLM returns, let's close the loop — execute the tool, send the result back, and repeat until the LLM stops calling tools.

In [15]:
def chat_with_tools(user_message: str, system: str = "You are a career counsellor.") -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    while True:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content

        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)

            print(f"  Tool called : {tc.function.name}")
            print(f"  Arguments   : {args}")

            if tc.function.name == "search_jobs":
                if result:
                    print(f"  Jobs found  : {len(result)}")
                    for i, job in enumerate(result, 1):
                        print(f"\n  ── Job {i} ──────────────────────────────────")
                        for k, v in job.items():
                            print(f"    {k:<14}: {v}")
                else:
                    print("  Jobs found  : 0 (no matches)")
            else:
                print(f"  Result      : {json.dumps(result)[:200]}...")

            print()
            messages.append({
                "role"        : "tool",
                "tool_call_id": tc.id,
                "content"     : json.dumps(result)
            })

print("chat_with_tools() ready.")


chat_with_tools() ready.


---
## Step 5 — Full Demo: GitHub + Job Search in One Shot

In [17]:
# Same question as Step 1 — but now the LLM has tools
print("=" * 55)
print("TOOL CALLS")
print("=" * 55)

answer = chat_with_tools(
    f"Candidate profile:\n{json.dumps(ARJUN, indent=2)}\n\n"
    "Check Arjun's GitHub (arjun91) and verify his technical claims. "
    "Then search for the 3 best matching jobs and briefly explain why each fits."
)

print("=" * 55)
print("LLM FINAL ANSWER")
print("=" * 55)
print(answer)

TOOL CALLS
  Tool called : read_github
  Arguments   : {'username': 'mahadevTW'}
  Result      : {"username": "mahadevTW", "name": "Mahadev Vyavahare", "bio": null, "company": null, "public_repos": 70, "followers": 28, "top_languages": ["HTML", "Jupyter Notebook", "JavaScript"], "top_repos": [{"n...

  Tool called : search_jobs
  Arguments   : {'skills': ['Java', 'Go', 'Kafka', 'Redis', 'distributed systems'], 'domain': 'fintech', 'min_years': 8}
  Jobs found  : 3

  ── Job 1 ──────────────────────────────────
    id            : 117
    job_uid       : fiserv:R-10391992
    job_title     : Manager, Software Development
    company       : Fiserv
    location      : Pune - Trion Business Park, India
    salary        : nan
    matched_skills: ['Java', 'Go', 'Kafka', 'distributed systems']
    match_score   : 80
    apply_link    : https://fiserv.wd5.myworkdayjobs.com/en-US/EXT/job/Pune---Trion-Business-Park-India/Manager--Software-Development_R-10391992
    description   : Calling all 

---
## The Aha Moment

**Without tools** — the LLM said: *"I don't have access to GitHub."*

**With tools** — the LLM:
1. Decided it needed GitHub data → called `read_github`
2. Decided it needed job matches → called `search_jobs`
3. Used both results to write a grounded, specific answer

Notice what *we* wrote vs what the *LLM* decided:

| We wrote | LLM decided |
|---|---|
| What tools exist (`TOOLS`) | Which tool to call |
| How tools work (`execute_tool`) | When to call it |
| The loop (`chat_with_tools`) | What arguments to pass |

**Thinking is the LLM's job. Acting is the tool's job. We just wire them together.**

---

> **Understand → Infer → *Act* → Plan → Explain**

---
## What's Next?

Right now we manually ask one question and get one answer.
But a real career coach doesn't wait to be asked — it **drives** the session:
parse resume → discover candidate → search jobs → run interview → produce a plan.

Something needs to decide the sequence, track the state, and loop until the goal is reached.

That's not a tool. That's an **Agent**.

→ **Notebook 04: The Agent Loop**